# Module 8: Spark UI Debugging Lab

**DSC 232R - Big Data Analysis Using Spark**

This hands-on lab covers:
1. Navigating the Spark UI
2. Interpreting stage timelines and task distributions
3. Identifying data skew and shuffle problems
4. Using the SQL tab for query plan analysis
5. Taking meaningful screenshots for documentation

## Learning Objectives

By the end of this lab, you will be able to:
- Navigate the Spark UI to identify performance bottlenecks
- Interpret stage timelines and task distributions
- Identify data skew and shuffle problems
- Use the SQL tab for query plan analysis
- Take meaningful screenshots for your project documentation

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum as spark_sum, rand, when
import numpy as np
import random
import time

# We won't use matplotlib for most of this lab - we'll use the actual Spark UI!

In [ ]:
# Create SparkSession with multiple cores to see parallelism
spark = SparkSession.builder \
    .appName("SparkUI-Debugging-Lab") \
    .master("local[4]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

# Get the Spark UI URL
spark_ui_url = spark.sparkContext.uiWebUrl
print(f"="*60)
print(f"SPARK UI AVAILABLE AT: {spark_ui_url}")
print(f"="*60)
print(f"\nOpen this URL in your browser to follow along with the lab.")
print(f"Keep this browser tab open throughout the exercises.")

---

## 1. Spark UI Overview

### Key Tabs

| Tab | What It Shows | When to Use |
|-----|--------------|-------------|
| **Jobs** | Overall job progress | Check job completion |
| **Stages** | Detailed stage info | Identify slow stages |
| **Storage** | Cached RDDs/DataFrames | Verify caching |
| **Environment** | Spark configuration | Debug config issues |
| **Executors** | Per-executor metrics | Identify executor problems |
| **SQL** | Query plans | Understand query execution |

---

## 2. Exercise 1: Understanding Job and Stage Structure

Let's create a query that generates multiple stages and observe it in the Spark UI.

In [ ]:
# Create sample data
print("Creating sample data...")
data = [(i, f"category_{i % 10}", np.random.random() * 100, i % 100)
        for i in range(500000)]

df = spark.createDataFrame(data, ["id", "category", "value", "group_id"])
print(f"Created DataFrame with {df.count()} rows")
print(f"Partitions: {df.rdd.getNumPartitions()}")

In [ ]:
# This query creates multiple stages
print("Running multi-stage query...")
print("Watch the Spark UI as this executes!")
print()

result = df.filter(col("value") > 10) \
           .groupBy("category") \
           .agg(
               count("*").alias("count"),
               avg("value").alias("avg_value")
           ) \
           .orderBy("count", ascending=False)

result.show()

### What to Observe in Spark UI

**Jobs Tab:**
1. Click on the job that just ran
2. Note how many stages it has
3. Each stage boundary = a shuffle

**Stages Tab:**
1. Find the stage with the longest duration
2. Click on it to see task details
3. Look at the "Event Timeline"

### Task: Take Screenshot 1

Capture a screenshot showing:
- All stages for your job
- Duration of each stage
- Shuffle read/write for each stage

In [ ]:
# Understanding the stage structure
print("Stage Structure Analysis")
print("=" * 50)
print("""
Expected stages for our query:

Stage 0: Read Data + Filter
  └── Input: DataFrame rows
  └── Output: Filtered rows (shuffle write)
  └── NO shuffle required yet

Stage 1: GroupBy + Aggregation  
  └── Input: Shuffle read (from Stage 0)
  └── SHUFFLE: Data redistributed by 'category' key
  └── Output: Aggregated results

Stage 2: OrderBy
  └── Input: Shuffle read (from Stage 1)  
  └── SHUFFLE: Data sorted globally
  └── Output: Final sorted results

Total Shuffles: 2 (Stage 0→1 and Stage 1→2)
""")

---

## 3. Exercise 2: Identifying Data Skew

Data skew is one of the most common performance problems in Spark. Let's create skewed data and observe it.

In [ ]:
# Create intentionally skewed data
# 90% of rows have category_0, 10% distributed among others

print("Creating skewed data (90% in one category)...")

skewed_data = []
for i in range(500000):
    if random.random() < 0.9:
        category = "hot_category"  # Hot key - 90% of data
    else:
        category = f"category_{random.randint(1, 99)}"
    skewed_data.append((i, category, random.random() * 100))

skewed_df = spark.createDataFrame(skewed_data, ["id", "category", "value"])

# Verify the skew
print("\nCategory distribution:")
skewed_df.groupBy("category").count().orderBy("count", ascending=False).show(5)

In [ ]:
# Run aggregation on skewed data
print("Running aggregation on skewed data...")
print("Watch the Spark UI - look for task duration differences!")
print()

skewed_result = skewed_df.groupBy("category") \
                         .agg(count("*").alias("count")) \
                         .orderBy("count", ascending=False)

skewed_result.show(5)

### Identifying Skew in Spark UI

**Stages Tab → Click on GroupBy Stage → Summary Metrics:**

Look for these signs of skew:
- **Task Duration**: Max >> Median
- **Shuffle Read Size**: Max >> Median  
- **GC Time**: High on specific tasks

**Event Timeline:**
- One task bar much longer than others
- Stragglers that hold up the whole stage

### Task: Take Screenshot 2

Capture a screenshot showing:
- Task duration distribution (histogram or timeline)
- Summary metrics with Max vs Median comparison

In [ ]:
# Analysis helper: What ratio indicates skew?
print("Data Skew Detection Guide")
print("=" * 50)
print("""
Skew Indicators in Task Metrics:

| Max/Median Ratio | Severity   | Action Needed           |
|------------------|------------|-------------------------|
| 1.0 - 1.5x       | Normal     | None                    |
| 1.5 - 3.0x       | Mild       | Monitor                 |
| 3.0 - 10x        | Moderate   | Consider optimization   |
| > 10x            | Severe     | Must fix (salting, etc) |

Common fixes for data skew:
1. Salting: Add random prefix to hot keys
2. Broadcast join: If one side is small
3. Adaptive Query Execution: Spark 3.0+
4. Repartition by different key
""")

---

## 4. Exercise 3: Comparing Join Strategies

The SQL tab shows you exactly what join strategy Spark chose.

In [ ]:
from pyspark.sql.functions import broadcast

# Large table (100K rows)
large_df = spark.createDataFrame(
    [(i, f"value_{i}", i % 100) for i in range(100000)],
    ["id", "data", "lookup_key"]
)

# Small lookup table (100 rows)
small_df = spark.createDataFrame(
    [(i, f"lookup_{i}") for i in range(100)],
    ["lookup_key", "lookup_value"]
)

print(f"Large table: {large_df.count()} rows")
print(f"Small table: {small_df.count()} rows")

In [ ]:
# Approach A: Regular join
print("Approach A: Regular Join")
print("Watch the Spark UI SQL tab!")
print()

result_a = large_df.join(small_df, "lookup_key")
count_a = result_a.count()
print(f"Result count: {count_a}")

In [ ]:
# Approach B: Broadcast join
print("Approach B: Broadcast Join")
print("Watch the Spark UI SQL tab - look for BroadcastHashJoin!")
print()

result_b = large_df.join(broadcast(small_df), "lookup_key")
count_b = result_b.count()
print(f"Result count: {count_b}")

### Analyzing in SQL Tab

**SQL Tab → Click on the query:**

For each approach, examine:
1. **Physical Plan**: Look for "BroadcastHashJoin" vs "SortMergeJoin"
2. **Stage count**: Broadcast should have fewer stages
3. **Shuffle bytes**: Broadcast should shuffle less

### Task: Take Screenshot 3

Capture screenshots showing:
- The physical plan for the regular join
- The physical plan for the broadcast join

In [ ]:
# Show the plans programmatically too
print("Regular Join Plan:")
print("=" * 50)
large_df.join(small_df, "lookup_key").explain()

print("\n" + "=" * 50)
print("\nBroadcast Join Plan:")
print("=" * 50)
large_df.join(broadcast(small_df), "lookup_key").explain()

---

## 5. Exercise 4: Memory and Spill Analysis

Let's create a scenario that may cause memory pressure and observe spilling.

In [ ]:
# Create a wider DataFrame that uses more memory
print("Creating wide DataFrame...")

# Generate data with many columns
wide_data = []
for i in range(200000):
    row = [i] + [f"col_{j}_val_{i}" for j in range(20)]
    wide_data.append(tuple(row))

columns = ["id"] + [f"col_{j}" for j in range(20)]
wide_df = spark.createDataFrame(wide_data, columns)

print(f"Wide DataFrame: {len(columns)} columns")

In [ ]:
# Aggregation that requires significant memory
print("Running memory-intensive aggregation...")
print("Watch the Executors tab for memory usage!")
print()

memory_result = wide_df.groupBy("col_0") \
                       .agg(*[count(f"col_{j}").alias(f"count_{j}")
                              for j in range(1, 20)])

memory_result.count()
print("Aggregation complete.")

### Checking for Memory Spill

**Stages Tab → Click on Stage → Summary Metrics:**

Look for:
- **Spill (Memory)**: Data that couldn't fit in memory
- **Spill (Disk)**: Spilled data written to disk

**Executors Tab:**
- Memory usage per executor
- Disk spill per executor

### Task: Take Screenshot 4

Capture a screenshot showing:
- Memory usage across executors
- Any spill metrics (Memory and Disk)

In [ ]:
print("Memory Spill Analysis Guide")
print("=" * 50)
print("""
What spill metrics mean:

Spill (Memory): Data that was evicted from execution memory
                but still held in storage memory before disk write.

Spill (Disk):   Data actually written to disk because memory
                was exhausted.

Impact:
- Memory spill: Moderate performance impact
- Disk spill:   Significant performance impact (10-100x slower)

Solutions:
1. Increase spark.executor.memory
2. Increase spark.memory.fraction
3. Reduce data per partition (more partitions)
4. Filter/aggregate data earlier in pipeline
""")

---

## 6. Exercise 5: Executor Analysis

The Executors tab shows health metrics for each executor.

In [ ]:
# Run a substantial workload to populate executor metrics
print("Running workload to populate executor metrics...")

# Create fresh data
workload_data = [(i, f"cat_{i % 50}", random.random() * 1000) 
                 for i in range(1000000)]
workload_df = spark.createDataFrame(workload_data, ["id", "category", "value"])

# Multiple operations to generate metrics
for iteration in range(3):
    workload_df.groupBy("category").agg(avg("value")).collect()
    print(f"  Iteration {iteration + 1} complete")

print("\nCheck the Executors tab now!")

### Executors Tab Deep Dive

Navigate to the **Executors** tab and examine:

| Metric | What to Look For |
|--------|------------------|
| **RDD Blocks** | Cached data distribution |
| **Storage Memory** | Memory used for caching |
| **Disk Used** | Spill to disk |
| **Active Tasks** | Should be balanced |
| **Failed Tasks** | Should be 0 |
| **Task Time** | Should be similar across executors |

### Healthy vs Unhealthy Patterns

**Healthy:**
- Similar task time across executors
- Low or no GC time
- No failed tasks
- Balanced storage memory usage

**Unhealthy:**
- One executor with much higher task time (skew)
- High GC time (memory pressure)
- Failed tasks (errors)
- Unbalanced memory usage

---

## 7. Debugging Checklist

Use this checklist when debugging Spark performance:

In [ ]:
debugging_checklist = """
SPARK PERFORMANCE DEBUGGING CHECKLIST
======================================

STAGE-LEVEL ANALYSIS
--------------------
[ ] Identify the slowest stage
[ ] Check shuffle read/write size
[ ] Look for spill (memory/disk)
[ ] Compare task durations (Max vs Median)

TASK-LEVEL ANALYSIS
-------------------
[ ] Check for stragglers (long-running tasks)
[ ] Look at task distribution across executors
[ ] Check GC time per task
[ ] Identify data skew patterns

QUERY PLAN ANALYSIS
-------------------
[ ] Review physical plan in SQL tab
[ ] Identify join strategies (Broadcast vs SortMerge)
[ ] Count number of shuffles (stage boundaries)
[ ] Look for unnecessary operations

EXECUTOR ANALYSIS
-----------------
[ ] Verify all executors are active
[ ] Check for failed tasks
[ ] Monitor memory usage
[ ] Look for imbalanced workload
"""

print(debugging_checklist)

---

## 8. Common Issues and Solutions

Quick reference for common Spark UI symptoms:

In [ ]:
import pandas as pd

issues_solutions = pd.DataFrame({
    'Spark UI Symptom': [
        'Only 1 executor active',
        'High GC time',
        'Large shuffle spill',
        'Single slow task',
        'Many failed tasks',
        'No tasks running'
    ],
    'Likely Cause': [
        'Local mode or config error',
        'Memory pressure',
        'Not enough memory for shuffle',
        'Data skew',
        'OOM errors',
        'Waiting for resources'
    ],
    'Solution': [
        'Check spark.executor.instances',
        'Increase spark.executor.memory',
        'Increase spark.memory.fraction',
        'Salting, repartition, or filter',
        'Reduce data per partition',
        'Check SLURM allocation'
    ]
})

print("Common Issues and Solutions")
print("=" * 80)
print(issues_solutions.to_string(index=False))

---

## 9. Required Screenshots for Your Project

For Milestone 3, include these Spark UI screenshots in your README:

### Screenshot A: Active Executors
- Shows multiple executors running
- Demonstrates distributed execution
- Location: Executors tab

### Screenshot B: Stage Metrics
- Shows your main processing stage
- Includes shuffle read/write
- Highlights any skew or spill

### Screenshot C: Query Plan (Optional but Recommended)
- Shows the physical plan for your main query
- Identifies join strategies
- From SQL tab

In [ ]:
readme_template = """
## Spark UI Verification (README Template)

### Executor Configuration
![Active Executors](images/spark_ui_executors.png)

Our job ran with [N] executors, each with [X]GB memory.
All executors were actively processing tasks.

### Stage Performance
![Stage Metrics](images/spark_ui_stages.png)

The [main operation] stage processed [X]GB with:
- Shuffle write: [X]GB
- Shuffle read: [X]GB  
- Max task duration: [X]s
- Median task duration: [X]s

### Analysis
The Max/Median ratio of [X] indicates [minimal/moderate/significant] data skew.
[No memory spill occurred / Memory spill of XMB occurred], confirming our executor 
memory configuration is [appropriate/needs adjustment] for the workload.

### Query Plan (Optional)
![Query Plan](images/spark_ui_sql.png)

Our main query uses [BroadcastHashJoin/SortMergeJoin] for the lookup table join.
[Describe any optimizations you made based on the query plan.]
"""

print(readme_template)

---

## 10. Practice: Document Your Current Session

Before finishing this lab, practice documenting what you've observed:

In [ ]:
# Run one final comprehensive query
print("Running final comprehensive query...")
print("Take your screenshots after this completes!")
print()

# Create meaningful data
final_data = [(i, f"customer_{i % 1000}", f"product_{i % 100}", 
               random.random() * 500, f"region_{i % 10}")
              for i in range(500000)]

sales_df = spark.createDataFrame(
    final_data, 
    ["sale_id", "customer_id", "product_id", "amount", "region"]
)

# Complex aggregation
final_result = sales_df.groupBy("region", "product_id") \
                       .agg(
                           count("*").alias("num_sales"),
                           spark_sum("amount").alias("total_amount"),
                           avg("amount").alias("avg_amount")
                       ) \
                       .orderBy("total_amount", ascending=False)

final_result.show(10)

print("\n" + "=" * 50)
print("NOW: Go to the Spark UI and take your screenshots!")
print("=" * 50)

### Your Documentation Exercise

Fill in this information based on what you observe in the Spark UI:

1. **Number of stages in the final query:** ___
2. **Slowest stage duration:** ___
3. **Total shuffle data:** ___
4. **Max/Median task duration ratio:** ___
5. **Any memory spill?** ___

---

## Summary

### Key Skills Learned

1. **Navigate Spark UI** tabs effectively
2. **Identify bottlenecks** in stages and tasks
3. **Detect data skew** through task duration analysis
4. **Compare join strategies** using query plans
5. **Monitor memory** and identify spill issues
6. **Document findings** with meaningful screenshots

### For Your Project

Include Spark UI analysis in your README showing:
- Confirmation of distributed execution (multiple executors)
- Performance metrics for your main stages
- Any issues identified and how you addressed them

In [ ]:
# Cleanup
spark.stop()
print("SparkSession stopped.")
print("\nNote: The Spark UI is no longer available after stopping the session.")

---

## Next: Framework Comparison (Spark vs Ray)

In Module 9, we'll compare Spark and Ray:
- When to choose each framework
- Performance comparison on the same task
- Integration patterns for using both together

See: `09_framework_comparison.ipynb`